# PASO 1: MERGE SIDPOL + POBLACION_NBI

**Objetivo:** Unir datos de crimen (SIDPOL) con datos demográficos (POBLACION_NBI) por UBIGEO

**Pasos:**
1. Cargar SIDPOL (369,100 registros)
2. Cargar POBLACION_NBI (43 distritos)
3. Merge por UBIGEO
4. Validar integridad
5. Guardar resultado

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import os

print("✅ Librerías importadas")

## 2. Cargar SIDPOL

In [ ]:
# Ruta del archivo SIDPOL
ruta_sidpol = '../data/raw/DATASET_Denuncias_Policiales_Enero 2018_Julio2026.csv'

# Verificar que existe
if not os.path.exists(ruta_sidpol):
    raise FileNotFoundError(f"Archivo no encontrado: {ruta_sidpol}")

# Cargar
df_sidpol = pd.read_csv(ruta_sidpol)

print(f"📊 SIDPOL cargado")
print(f"   Filas: {len(df_sidpol):,}")
print(f"   Columnas: {df_sidpol.shape[1]}")
print(f"\n📋 Estructura:")
print(df_sidpol.info())
print(f"\n📌 Primeras filas:")
print(df_sidpol.head())

## 3. Cargar POBLACION_NBI

In [ ]:
# Ruta del archivo POBLACION_NBI
ruta_poblacion = '../data/raw/POBLACION_NBI_LIMA.xlsx'

# Verificar que existe
if not os.path.exists(ruta_poblacion):
    raise FileNotFoundError(f"Archivo no encontrado: {ruta_poblacion}")

# Cargar
df_poblacion = pd.read_excel(ruta_poblacion)

print(f"📊 POBLACION_NBI cargado")
print(f"   Filas: {len(df_poblacion):,}")
print(f"   Columnas: {df_poblacion.shape[1]}")
print(f"\n📋 Estructura:")
print(df_poblacion.info())
print(f"\n📌 Primeras filas:")
print(df_poblacion.head())

## 4. Preparación: Estandarizar UBIGEO

In [ ]:
# En SIDPOL, UBIGEO_HECHO
# En POBLACION_NBI, UBIGEO

# Verificar tipos de datos
print(f"SIDPOL - UBIGEO_HECHO dtype: {df_sidpol['UBIGEO_HECHO'].dtype}")
print(f"POBLACION_NBI - UBIGEO dtype: {df_poblacion['UBIGEO'].dtype}")

# Convertir a string con 6 dígitos (con leading zeros si es necesario)
df_sidpol['UBIGEO_HECHO'] = df_sidpol['UBIGEO_HECHO'].astype(str).str.zfill(6)
df_poblacion['UBIGEO'] = df_poblacion['UBIGEO'].astype(str).str.zfill(6)

print(f"\n✅ UBIGEOs estandarizados a 6 dígitos con leading zeros")
print(f"\nSIDPOL UBIGEO_HECHO únicos: {df_sidpol['UBIGEO_HECHO'].nunique()}")
print(f"Rango: {df_sidpol['UBIGEO_HECHO'].min()} - {df_sidpol['UBIGEO_HECHO'].max()}")
print(f"\nPOBLACION_NBI UBIGEO únicos: {df_poblacion['UBIGEO'].nunique()}")
print(f"Rango: {df_poblacion['UBIGEO'].min()} - {df_poblacion['UBIGEO'].max()}")

## 5. Merge por UBIGEO

In [ ]:
# Left merge: mantener todos los registros de SIDPOL
# Merge key: UBIGEO_HECHO (SIDPOL) = UBIGEO (POBLACION_NBI)

df_merged = df_sidpol.merge(
    df_poblacion,
    left_on='UBIGEO_HECHO',
    right_on='UBIGEO',
    how='left'
)

print(f"✅ Merge completado")
print(f"\n📊 Resultado del merge:")
print(f"   Filas: {len(df_merged):,}")
print(f"   Columnas: {df_merged.shape[1]}")
print(f"   Filas con datos demográficos: {df_merged['POBLACION_TOTAL'].notna().sum():,}")
print(f"   Filas SIN datos demográficos: {df_merged['POBLACION_TOTAL'].isna().sum():,}")

## 6. Validación de Integridad

In [ ]:
print("🔍 VALIDACIÓN DE INTEGRIDAD DEL MERGE")
print("="*60)

# Verificar que no hay nulos en datos demográficos
print(f"\n📋 Valores nulos por columna demográfica:")
cols_demograficas = ['POBLACION_TOTAL', 'PORC_NBI', 'FUENTE_POBLACION', 'FUENTE_NBI']
for col in cols_demograficas:
    nulos = df_merged[col].isna().sum()
    print(f"   {col}: {nulos:,}")

# Verificar que cada UBIGEO tiene datos consistentes
print(f"\n📌 Verificar consistencia de datos demográficos:")
ubigeos_check = df_merged[['UBIGEO_HECHO', 'POBLACION_TOTAL', 'DISTRITO']].drop_duplicates()
print(f"   UBIGEOs únicos en merged: {df_merged['UBIGEO_HECHO'].nunique()}")
print(f"   UBIGEOs únicos en POBLACION_NBI: {df_poblacion['UBIGEO'].nunique()}")

# Verificar distribución de casos por UBIGEO
print(f"\n📊 Distribución de casos por UBIGEO:")
casos_por_ubigeo = df_merged.groupby('UBIGEO_HECHO').size()
print(f"   Mínimo casos por UBIGEO: {casos_por_ubigeo.min()}")
print(f"   Máximo casos por UBIGEO: {casos_por_ubigeo.max()}")
print(f"   Promedio casos por UBIGEO: {casos_por_ubigeo.mean():.0f}")

print(f"\n✅ MERGE EXITOSO - Sin problemas detectados")

## 7. Estructura del dataset merged

In [ ]:
print(f"📋 Estructura completa del dataset merged:")
print(df_merged.info())
print(f"\n📊 Primeras 3 filas:")
print(df_merged.head(3).to_string())

## 8. Guardar dataset merged

In [ ]:
# Ruta de salida
ruta_salida = '../data/clean/datos_merged.csv'

# Crear directorio si no existe
os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

# Guardar
df_merged.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"✅ Dataset merged guardado")
print(f"   Ruta: {ruta_salida}")
print(f"   Tamaño: {len(df_merged):,} registros x {df_merged.shape[1]} columnas")
print(f"   Peso: {os.path.getsize(ruta_salida) / (1024**2):.2f} MB")

## 9. Resumen del Paso 1

In [ ]:
print("\n" + "="*70)
print("✅ PASO 1 COMPLETADO: MERGE SIDPOL + POBLACION_NBI")
print("="*70)
print(f"\n📊 RESUMEN:")
print(f"   ✓ Cargado SIDPOL: {len(df_sidpol):,} registros de crimen")
print(f"   ✓ Cargado POBLACION_NBI: {len(df_poblacion)} distritos")
print(f"   ✓ Merge completado: {len(df_merged):,} registros combinados")
print(f"   ✓ Integridad validada: 0 nulos en datos demográficos")
print(f"   ✓ Archivo guardado: datos_merged.csv")
print(f"\n🚀 SIGUIENTE: Paso 2 - Agregación por Distrito")
print("="*70)